In [1]:
from getpass import getpass
import sys, os

REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"
LOCAL_REPO_PATH = f"/content/{REPO_NAME}"

# Mount Google Drive first (Colab only)
if '/content' in os.getcwd():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# Clone only if not existing
if not os.path.exists(LOCAL_REPO_PATH):
    print("Cloning repo...")
    token = getpass("GitHub Token: ")
    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir("/content")

# Enable import of your modules
if LOCAL_REPO_PATH not in sys.path:
    os.chdir(LOCAL_REPO_PATH)
    sys.path.append(os.getcwd())
    os.chdir("/content")

Mounted at /content/drive
Cloning repo...
GitHub Token: ··········
Cloning into 'RecSys-Challenge-2025'...
remote: Enumerating objects: 225, done.
remote: Counting objects: 100% (225/225), done.
remote: Compressing objects: 100% (172/172), done.
remote: Total 225 (delta 67), reused 202 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (225/225), 7.34 MiB | 23.19 MiB/s, done.
Resolving deltas: 100% (67/67), done.


In [3]:
!pip install optuna
import optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 8.8 MB/s eta 0:00:00


In [4]:
import importlib
import scipy.sparse as sps

from Challenge import paths
importlib.reload(paths)

from Evaluation.Evaluator import EvaluatorHoldout
from Challenge.hyper_tuning import hyperparameter_tuning

Running on: colab — BASE_DIR = /content/drive/MyDrive/RecSys/demo


In [5]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [6]:
# Set up evaluator
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[10])

EvaluatorHoldout: Ignoring 76 ( 0.1%) Users that have less than 1 test interactions


In [7]:
# Define objective function for hyperparameter tuning
from Recommenders.NonPersonalizedRecommender import GlobalEffects

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = GlobalEffects(URM_train)
    recommender_instance.fit(
        # shrink factor
        lambda_user=optuna_trial.suggest_int("lambda_user", 0, 1000),
        lambda_item=optuna_trial.suggest_int("lambda_item", 0, 1000)
    )

    result_df, _ = evaluator.evaluateRecommender(recommender_instance)

    return result_df.loc[10]["MAP"]

In [8]:
save_results, optuna_study = hyperparameter_tuning(objective_function, n_trials=50)

[I 2025-11-02 21:57:31,444] A new study created in memory with name: no-name-48724ff1-1059-4709-875c-1b185d950b6d


GlobalEffectsRecommender: URM Detected 25 ( 0.2%) items with no interactions.
EvaluatorHoldout: Processed 69802 (100.0%) in 23.76 sec. Users per second: 2938


[I 2025-11-02 21:57:55,568] Trial 0 finished with value: 0.042565669310424305 and parameters: {'lambda_user': 280, 'lambda_item': 597}. Best is trial 0 with value: 0.042565669310424305.


GlobalEffectsRecommender: URM Detected 25 ( 0.2%) items with no interactions.


/content/RecSys-Challenge-2025/Challenge/hyper_tuning.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.results_df = pd.concat([self.results_df, pd.DataFrame([hyperparam_dict])], ignore_index=True)


...

GlobalEffectsRecommender: URM Detected 25 ( 0.2%) items with no interactions.
EvaluatorHoldout: Processed 69802 (100.0%) in 24.30 sec. Users per second: 2873


[I 2025-11-02 22:17:22,261] Trial 48 finished with value: 0.04525450958106751 and parameters: {'lambda_user': 22, 'lambda_item': 788}. Best is trial 21 with value: 0.04773514983197245.


GlobalEffectsRecommender: URM Detected 25 ( 0.2%) items with no interactions.
EvaluatorHoldout: Processed 69802 (100.0%) in 24.28 sec. Users per second: 2874


[I 2025-11-02 22:17:46,993] Trial 49 finished with value: 0.047426342106901644 and parameters: {'lambda_user': 312, 'lambda_item': 951}. Best is trial 21 with value: 0.04773514983197245.


Study statistics: 
  Number of finished trials:  50
  Number of pruned trials:  0
  Number of complete trials:  50
Best trial:
  Value Validation:  0.04773514983197245
  Params:  {'lambda_user': 0, 'lambda_item': 1000}
All results:
      result  lambda_user  lambda_item
0   0.042566        280.0        597.0
1   0.045844        894.0        860.0
2   0.047415        356.0        948.0
3   0.035314        959.0        167.0
4   0.035537        104.0        175.0
5   0.034016        757.0        114.0
6   0.037056        732.0        264.0
7   0.035310        332.0        171.0
8   0.040270        295.0        483.0
9   0.038308        726.0        357.0
10  0.047488          1.0        976.0
11  0.047488         62.0        972.0
12  0.044850         13.0        766.0
13  0.047556        155.0        984.0
14  0.043874        175.0        704.0
15  0.045796        501.0        829.0
16  0.043363        171.0        653.0
17  0.047715        470.0        998.0
18  0.040261        498.0  

In [9]:
# Train final model on train + validation with best hyperparameters
recommender = GlobalEffects(URM_train)
recommender.fit(
    lambda_user=optuna_study.best_trial.params["lambda_user"],
    lambda_item=optuna_study.best_trial.params["lambda_item"]
)

GlobalEffectsRecommender: URM Detected 25 ( 0.2%) items with no interactions.


In [10]:
evaluator.evaluateRecommender(recommender)

EvaluatorHoldout: Processed 69802 (100.0%) in 24.50 sec. Users per second: 2849


(       PRECISION PRECISION_RECALL_MIN_DEN    RECALL       MAP MAP_MIN_DEN  \
 cutoff                                                                      
 10      0.089046                  0.09466  0.036598  0.047735     0.05044   
 
              MRR      NDCG        F1  HIT_RATE ARHR_ALL_HITS  ...  \
 cutoff                                                        ...   
 10      0.235025  0.090918  0.051875  0.450001      0.324252  ...   
 
        COVERAGE_USER COVERAGE_USER_HIT USERS_IN_GT DIVERSITY_GINI  \
 cutoff                                                              
 10          0.998912          0.449512    0.998912       0.001363   
 
        SHANNON_ENTROPY RATIO_DIVERSITY_HERFINDAHL RATIO_DIVERSITY_GINI  \
 cutoff                                                                   
 10            4.114612                   0.932489             0.006999   
 
        RATIO_SHANNON_ENTROPY RATIO_AVERAGE_POPULARITY RATIO_NOVELTY  
 cutoff                                   

In [11]:
# Train final model on train + validation with best hyperparameters
recommender2 = GlobalEffects(URM_train)
recommender2.fit()
evaluator.evaluateRecommender(recommender2)

GlobalEffectsRecommender: URM Detected 25 ( 0.2%) items with no interactions.
EvaluatorHoldout: Processed 69802 (100.0%) in 24.82 sec. Users per second: 2812


(       PRECISION PRECISION_RECALL_MIN_DEN    RECALL       MAP MAP_MIN_DEN  \
 cutoff                                                                      
 10      0.051314                 0.055354  0.023538  0.030854    0.033276   
 
              MRR      NDCG        F1  HIT_RATE ARHR_ALL_HITS  ...  \
 cutoff                                                        ...   
 10      0.207523  0.065712  0.032272  0.335979      0.249896  ...   
 
        COVERAGE_USER COVERAGE_USER_HIT USERS_IN_GT DIVERSITY_GINI  \
 cutoff                                                              
 10          0.998912          0.335613    0.998912       0.001197   
 
        SHANNON_ENTROPY RATIO_DIVERSITY_HERFINDAHL RATIO_DIVERSITY_GINI  \
 cutoff                                                                   
 10            3.864919                    0.92351             0.006144   
 
        RATIO_SHANNON_ENTROPY RATIO_AVERAGE_POPULARITY RATIO_NOVELTY  
 cutoff                                   

In [12]:
optuna.visualization.plot_optimization_history(optuna_study)

In [13]:
optuna.visualization.plot_param_importances(optuna_study)